## 10. 모델 평가

In [10]:
# 저장된 모델 및 테스트 데이터 불러오기
import numpy as np
from tensorflow.keras.models import load_model

model = load_model('./model/best_model_article.h5')

input_X_test = np.load('./data/article_X_test.npy')
input_y_test = np.load('./data/article_y_test.npy')

len(input_X_test), len(input_y_test)

(121478, 121478)

In [11]:
test_loss, test_acc = model.evaluate(input_X_test, input_y_test)

3797/3797 [==============================] - 109s 29ms/step - loss: 0.4957 - accuracy: 0.7612


In [12]:
print(f'정확도 : {test_acc * 100:.2f}')

정확도 : 76.12


In [13]:
#   predict: 테스트 데이터 예측값 구하기
results = model.predict(input_X_test)

3797/3797 [==============================] - 101s 27ms/step


In [14]:
print(results[:1])

[[0.01951736 0.9804827 ]]


In [15]:
labels = ['하락', '상승']
user_outputs = [labels[np.argmax(result)] for result in results] 
print(user_outputs[:10])

['상승', '상승', '상승', '상승', '하락', '상승', '상승', '상승', '하락', '상승']


In [16]:
#   평가 결과 확인: sklearn의 classification_report
from sklearn.metrics import classification_report

print(classification_report(
    #   1차원 배열로 전환
    np.argmax(input_y_test, axis = 1), 
    np.argmax(results, axis = 1)))

              precision    recall  f1-score   support

           0       0.76      0.71      0.73     55958
           1       0.76      0.81      0.78     65520

    accuracy                           0.76    121478
   macro avg       0.76      0.76      0.76    121478
weighted avg       0.76      0.76      0.76    121478



## 11. 예측

In [19]:
import joblib
import pandas as pd
from kiwipiepy import Kiwi
from tensorflow.keras.preprocessing.sequence import pad_sequences

model = load_model('./model/best_model_article.h5')
#   이전 파일(감성분석_260529_lstm_model_training)에서 학습한 tokenizer_001 저장 및 불러오기
tokenizer_001 = joblib.load('./model/article_tokenizer.pkl')

stnd_poses = ['NNG', 'NNP', 'VV', 'VA']
stop_words = []

t = Kiwi()
max_len = 700

#   압룍된 기사에 대한 긍부정 판단 함수
def predict_direction(text):
    tokens = [token.form for token in t.tokenize(text)
                             if token.tag in stnd_poses
                                and token.form not in stop_words]#  형태소 분석
    encoded_tokens = tokenizer_001.texts_to_sequences([tokens])
    X = pad_sequences(encoded_tokens, maxlen = max_len)
    #   예측
    results = model.predict(X, verbose = 0)
    labels = ['하락', '상승']
    index = np.argmax(results[0])
    user_output = labels[index]
    return user_output, results[0][index]


In [20]:
import pandas as pd
#   테스트
test_001 = pd.read_csv('./data/naver_economic_news_260524_260531.csv')

test_002 = test_001[test_001['일자'] == '2026-05-31'].head()
articles = test_002['본문'].tolist()

for article in articles:
    user_output, prob = predict_direction(article)
    
    print(f'예측 결과: {user_output} ({prob * 100:.2f}%)')
    print(f'기사 내용: {article[:50]}')

예측 결과: 하락 (99.74%)
기사 내용: 일론 머스크와 스페이스X 로고 [로이터] [헤럴드경제=문이림 기자] 스페이스X가 다음달 1
예측 결과: 상승 (64.83%)
기사 내용: 크리스토퍼 월러 연방준비제도(Fed) 이사가 스테이블코인 확산이 미국 통화정책의 글로벌 영
예측 결과: 하락 (71.18%)
기사 내용: ‘한국의 위대한 투자자’ 김태홍 그로쓰힐자산운용 대표 인터뷰 매경플러스 ‘한국의 위대한 투
예측 결과: 상승 (99.87%)
기사 내용: (서울=연합뉴스) 홍국기 기자 = 오케스트라프라이빗에쿼티(오케스트라PE)는 매머드커피와 서
예측 결과: 하락 (80.78%)
기사 내용: ※ 2008년 8월 ‘홍길용의 머니스토리’로 시작해 ‘홍길용의 화식열전’으로 이름을 바꾸며
